In [ ]:
# !conda create -n skmob python=3.9.0
# !conda install conda-forge scikit-mobility

In [1]:
import networkx as nx
import pandas as pd # version 2.3.1
import geopandas as gpd #1.0.1
import numpy as np # 2.2.6
import shapely   # 2.0.7 , measure importance of features through SHAP values
import skmob

Cannot find header.dxf (GDAL_DATA is not defined)


In [2]:
import os

data_folder = "data/2017-2021"
# datasets is aggregated commuter trajectories
datasets = {}

for filename in os.listdir(data_folder):
    # filename.split("_") creates list of items split from the filename -> [-1] access last item -> .split(".") splits last item by "."-> [0] accesses the first item which is the state code
    state_code = filename.split("_")[-1].split(".")[0]
    file_path = os.path.join(data_folder, filename)
    df = pd.read_csv(file_path)
    # Add the DataFrame to the container with the state code as the label
    datasets[state_code] = df
    

In [33]:
datasets['FL'].groupby("Otract").nunique()

,Dtract,EST
Otract,,
12001000201,24,15
12001000202,28,15
12001000301,27,19
12001000302,27,13
12001000400,28,16
...,...,...
12133970104,14,10
12133970200,27,10
12133970301,25,12


In [6]:
len(datasets['FL']['Dtract'].unique())

8253

In [4]:
len(datasets['FL']['Otract'].unique())

5075

In [7]:
set(datasets['FL']['Otract'].unique()) - set(datasets['FL']['Dtract'].unique())

{np.int64(12071010107), np.int64(12083000806), np.int64(12099005943)}

In [ ]:
set(datasets['FL']['Dtract'].unique()) - set(datasets['FL']['Otract'].unique())
# len(set(datasets['FL']['Dtract'].unique()) - set(datasets['FL']['Otract'].unique()))

{np.int64(36071014400),
 np.int64(6025011201),
 np.int64(17197883402),
 np.int64(51700031500),
 np.int64(19153010707),
 np.int64(13051002900),
 np.int64(48491020310),
 np.int64(13081010201),
 np.int64(36011040800),
 np.int64(13127000101),
 np.int64(13127000103),
 np.int64(34013020200),
 np.int64(34017017900),
 np.int64(19113001005),
 np.int64(34039038000),
 np.int64(17031831600),
 np.int64(45063020601),
 np.int64(45063020605),
 np.int64(36047011901),
 np.int64(48491020352),
 np.int64(21111980100),
 np.int64(39153532999),
 np.int64(42101014600),
 np.int64(36103135303),
 np.int64(48113016903),
 np.int64(36119011401),
 np.int64(51107610702),
 np.int64(17043841102),
 np.int64(27147960400),
 np.int64(6085509201),
 np.int64(37083930702),
 np.int64(17043841111),
 np.int64(9001050200),
 np.int64(26125197400),
 np.int64(36047044700),
 np.int64(35013001309),
 np.int64(17037000800),
 np.int64(13095010401),
 np.int64(29189218401),
 np.int64(12009980000),
 np.int64(1097007204),
 np.int64(3400304240

In [15]:
datasets['FL']

,Otract,Dtract,EST
0,12001000201,12001000201,405.0
1,12001000201,12001000202,40.0
2,12001000201,12001000301,20.0
3,12001000201,12001000400,225.0
4,12001000201,12001000500,35.0
...,...,...,...
220720,12133970303,12133970103,55.0
220721,12133970303,12133970104,105.0
220722,12133970303,12133970200,15.0
220723,12133970303,12133970302,50.0


In [17]:
# datasets['FL']['Otract' = np.int64(12071010107)]['EST']
datasets['FL'].loc[datasets['FL']['Otract'] == 12071010107]

,Otract,Dtract,EST
83194,12071010107,2020002301,30.0
83195,12071010107,12015010301,60.0
83196,12071010107,12015010302,10.0
83197,12071010107,12015020303,4.0
83198,12071010107,12071000600,15.0
83199,12071010107,12071000700,30.0
83200,12071010107,12071000800,20.0
83201,12071010107,12071001104,15.0
83202,12071010107,12071001203,20.0
83203,12071010107,12071001207,25.0


In [18]:
datasets['FL'].loc[datasets['FL']['Dtract'] == 12071010107]

,Otract,Dtract,EST


Some census tracts are listed only as an origin and not as a destination census tract. This is interpreted as there being no commuter workers within those census tracts. Additionally, some census tracts are listed only as destinations and not as origin census tracts. This is interpreted as the resident population is 0. 

We will calculate the Population of a census tract as the sum of travelers when grouped by an Otract id matching the census tract id.
The Number of Jobs held by commuters is the sum of travelers when grouped by a Dtract id matching the census tract id. For 

In [25]:
fl_data = datasets['FL']

In [26]:
no_jobs = set(datasets['FL']['Otract'].unique()) - set(datasets['FL']['Dtract'].unique())

In [27]:
no_jobs = pd.DataFrame(no_jobs, columns = ['Id'])
no_jobs

,Id
0,12071010107
1,12083000806
2,12099005943


In [28]:
dtracts = datasets['FL']['Dtract'].unique()
dtracts = pd.DataFrame(dtracts, columns = ['Id'])

In [29]:
tracts = pd.concat([no_jobs, dtracts])

In [42]:
tracts.loc[tracts['Id'].isin(no_jobs['Id']), 'Jobs'] = 0

In [31]:
fl_data['Dtract'].unique()

array([12001000201, 12001000202, 12001000301, ...,  1101003301,
       12133970102,  9013881300])

In [36]:
jobs

Dtract
1003010100      10.0
1003010200      10.0
1003010400     105.0
1003010500     140.0
1003010704      20.0
               ...  
55139001700      4.0
55141010400      4.0
55141011100     10.0
56005000300     10.0
56033000400     15.0
Name: EST, Length: 8253, dtype: float64

In [40]:
jobs = fl_data.groupby('Dtract')['EST'].sum()
tracts['Id'].map(jobs)

0          NaN
1          NaN
2          NaN
0       2603.0
1       1210.0
         ...  
8248      45.0
8249      20.0
8250     100.0
8251      60.0
8252       4.0
Name: Id, Length: 8256, dtype: float64

In [43]:
tracts['Jobs']

0          0.0
1          0.0
2          0.0
0       2603.0
1       1210.0
         ...  
8248      45.0
8249      20.0
8250     100.0
8251      60.0
8252       4.0
Name: Jobs, Length: 8256, dtype: float64

In [44]:
grouped_dsets = {}

In [45]:
for (state_code, df) in enumerate(datasets):
    grouped = pd.DataFrame()
    # no_jobs is a datafram of tracts that have a resident population, but don't provide any jobs for commuting workers from other census tracts
    no_jobs = pd.DataFrame(set(df['Otract'].unique()) - set(df['Dtract'].unique()), columns = ['Id'])
    # no_pop is a data frame of tracts that provide jobs,  but do not have any resident population
    no_pop = pd.DataFrame(set(df['Dtract'].unique()) - set(df['Otract'].unique()), columns = ['Id'])
    
    grouped['Id'] = df['Dtract'].unique()
    # now add the Ids of the census tracts that are only origin tracts and don't function as destination tract
    grouped = pd.concat([grouped, no_jobs])

    # Now add null job values to census tracts that are only origin tracts
    grouped.loc[grouped['Id'].isin(no_jobs['Id']), 'Jobs'] = 0
    # add job values to all other tracts with jobs
    jobs = df.groupby(df['Dtract'].unique())['EST'].sum()
    grouped['Id'].map(jobs)

    # Add null pop values to census tracts that are only destination tracts
    grouped.loc[grouped['Id'].isin(no_pop['Id']), 'Population'] = 0
    # add pop values to all other tracts with pop
    pop = df.groupby(df['Otract'].unique())['EST'].sum()
    grouped['Population'].map(pop)
    grouped_dsets[state_code] = grouped

TypeError: string indices must be integers

In [32]:
datasets['FL']

,Otract,Dtract,EST
0,12001000201,12001000201,405.0
1,12001000201,12001000202,40.0
2,12001000201,12001000301,20.0
3,12001000201,12001000400,225.0
4,12001000201,12001000500,35.0
...,...,...,...
220720,12133970303,12133970103,55.0
220721,12133970303,12133970104,105.0
220722,12133970303,12133970200,15.0
220723,12133970303,12133970302,50.0


In [ ]:
assert(datasets['AK'])

### Now add other predictors used by the traditional gravity model: :
- longitude, latitude of the centroid of each census tract


### Now add additional predictors to use:
- SVI info for each census tract
- POI_lon, POI_lat where POI is the some POI for each census tract
- POI_text which is textual data about the rating, review of the POI